# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook guides users through the process of loading, exploring, and analyzing the FAIR² dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

We first enumerate the record sets (tables) and then show their fields and columns, all referenced by their `@id`.

In [ ]:
# List all record sets and their field/column IDs
print('Available record sets in the dataset:')
record_sets = []
for rset in getattr(metadata, 'recordSet', []):
    print(f"- @id: {getattr(rset, '@id', None)} | name: {getattr(rset, 'name', None)}")
    record_sets.append(getattr(rset, '@id', None))
    if hasattr(rset, 'field'):
        print('    Fields:')
        for f in rset.field:
            print(f"      - @id: {getattr(f, '@id', None)} | name: {getattr(f, 'name', None)}")
    if hasattr(rset, 'column'):
        print('    Columns:')
        for c in rset.column:
            print(f"      - @id: {getattr(c, '@id', None)} | name: {getattr(c, 'name', None)}")
print('\nIf the above is empty or only contains None values, please check the dataset schema or metadata.')

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# If there are no record sets in metadata.recordSet, try extracting from the default record set.
# Otherwise, use the discovered record set @id(s):

if len(record_sets) == 0:
    # Try an auto-discovery: get one record set via dataset.records()
    try:
        print('No record sets found in metadata. Attempting automatic discovery...')
        # List the first 3 records to explore structure
        records_iterator = dataset.records()
        records = []
        for i, rec in enumerate(records_iterator):
            records.append(rec)
            if i == 2:
                break
        df = pd.DataFrame(records)
        print('Sample columns:', df.columns.tolist())
        display(df.head())
        first_record_set_id = None
        dataframes = {'default': df}
    except Exception as e:
        print('Could not automatically extract records. Error:', str(e))
else:
    # Extract all record set tables
    dataframes = {}
    for record_set_id in record_sets:
        print(f'Extracting record set: {record_set_id}')
        try:
            records = list(dataset.records(record_set=record_set_id))
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f'  Columns for {record_set_id}:', df.columns.tolist())
        except Exception as e:
            print(f'  Failed to extract {record_set_id}:', str(e))

    # Use the first record set for demonstration
    first_record_set_id = record_sets[0]
    print('\nPreview of records for record set:', first_record_set_id)
    display(dataframes[first_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

We'll:
- Choose a numeric field by `@id` (if present)
- Filter records with a threshold
- Normalize the numeric field
- Optionally group by a categorical field referenced by its `@id`

In [ ]:
# Try automatic selection of numeric and categorical fields by dtype
import numpy as np

if len(dataframes) > 0:
    if 'default' in dataframes:
        df = dataframes['default']
        record_set_key = 'default'
    else:
        record_set_key = list(dataframes.keys())[0]
        df = dataframes[record_set_key]

    numeric_fields = df.select_dtypes(include=[np.number]).columns.tolist()
    print('Numeric fields in the DataFrame:', numeric_fields)

    if numeric_fields:
        numeric_field = numeric_fields[0]  # use the first numeric col
        threshold = df[numeric_field].mean() if df[numeric_field].notnull().sum() > 0 else 0
        filtered_df = df[df[numeric_field] > threshold]
        print(f"\nFiltered records with {numeric_field} > {threshold:.2f}:")
        display(filtered_df.head())

        norm_col = f"{numeric_field}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"\nNormalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, norm_col]].head())

        # Try grouping by a categorical column (besides the numeric field)
        cat_fields = df.select_dtypes(include=['object', 'category']).columns.difference([numeric_field]).tolist()
        if cat_fields:
            group_field = cat_fields[0]
            print(f"\nGrouping data by '{group_field}' and computing means of numeric columns:")
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            display(grouped_df.head())
        else:
            print('No categorical fields available for grouping.')
    else:
        print('No numeric fields available for EDA.')
else:
    print('No dataframes loaded yet.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We plot the distribution of the numeric field and, if grouped, compare group means.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if len(dataframes) > 0 and 'numeric_field' in locals() and numeric_field in df.columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=15)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    if 'group_field' in locals() and group_field in df.columns:
        plt.figure(figsize=(10, 4))
        sns.boxplot(x=group_field, y=numeric_field, data=filtered_df)
        plt.title(f'{numeric_field} by {group_field}')
        plt.show()
else:
    print('No suitable data for visualization found (check numeric_field/group_field).')

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- We successfully loaded the FAIR² dataset using its Croissant schema.
- We explored the available record sets and fields via their `@id`s.
- Data was extracted into DataFrames for analysis.
- Exploratory analysis was performed on numeric and categorical fields.
- Visualizations provided insight into variable distribution and group differences.

**Next steps:** You can further explore other fields and record sets using their `@id` references, conduct statistical tests, or build machine learning models on the extracted DataFrames.